In [1]:
!pip install langchain langchain_groq langchain_community langchain_core \
             langchain_huggingface langchain_chroma faiss-cpu chromadb \
             sentence-transformers scikit-learn matplotlib numpy --quiet
print("✅ All packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)
print(llm.invoke("Reply with exactly: LLM_OK").content)

LLM_OK


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

# Downloads ~90 MB once, then cached locally — no API key needed
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
    # Produces 384-dimensional vectors; fast on CPU
)
print("✅ Embedding model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


RAW_TEXT= """
## Retrieval-Augmented Generation
Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a language model """
RAW_TEXT += """to produce grounded, accurate answers. Instead of relying solely on training data, RAG retrieves """
RAW_TEXT += """relevant documents from a knowledge base and includes them in the prompt. """
RAW_TEXT += """This allows the model to answer questions about unseen data, reduces hallucinations, """
RAW_TEXT += """and makes answers verifiable. The pipeline has two phases: indexing (embed and store) """
RAW_TEXT += """and querying (retrieve and generate).
"""


print(f'Knowledge base: {len(RAW_TEXT)} characters')

Knowledge base: 522 characters


In [18]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=['\n\n', '\n', '. ', ' ', '']
)

chunks = splitter.create_documents([RAW_TEXT])
print(f'Split into {len(chunks)} chunks\n')

for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i} ({len(chunk.page_content)} chars) ---')
    print(chunk.page_content, '...')
    print()

Split into 3 chunks

--- Chunk 0 (33 chars) ---
## Retrieval-Augmented Generation ...

--- Chunk 1 (276 chars) ---
Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a language model to produce grounded, accurate answers. Instead of relying solely on training data, RAG retrieves relevant documents from a knowledge base and includes them in the prompt ...

--- Chunk 2 (210 chars) ---
. This allows the model to answer questions about unseen data, reduces hallucinations, and makes answers verifiable. The pipeline has two phases: indexing (embed and store) and querying (retrieve and generate). ...



In [15]:
from langchain_community.vectorstores import FAISS

# Build the index — embeds every chunk automatically
faiss_store = FAISS.from_documents(chunks, embeddings)

print('✅ FAISS index built')
print(f'   Vectors stored : {faiss_store.index.ntotal}')
print(f'   Dimensions     : {faiss_store.index.d}')

✅ FAISS index built
   Vectors stored : 6
   Dimensions     : 384


In [28]:
# Basic similarity search — returns top-k most relevant chunks
query   = 'What is RAG and why is it useful?'
results = faiss_store.similarity_search(query, k=3)

print(f'Query: {query}\n')
for i, doc in enumerate(results):
    print(f'--- Result {i+1} [topic: {doc.metadata.get("topic")}] ---')
    print(doc.page_content)
    print()

Query: What is RAG and why is it useful?

--- Result 1 [topic: None] ---
Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a language model to produce grounded, accurate answers. Instead of relying solely on training data, RAG retrieves relevant documents from a knowledge base and includes them in the prompt

--- Result 2 [topic: None] ---
. Its ecosystem of third-party packages makes it ideal for web development, data science, machine learning, and automation. The package manager pip provides access to over 400,000 packages.

--- Result 3 [topic: None] ---
Python is a high-level, interpreted programming language created by Guido van Rossum in 1991. It emphasises code readability and uses significant indentation. Python supports multiple paradigms including procedural, object-oriented, and functional programming



In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Your text data defined exactly as requested
RAW_TEXT = """Retrieval-Augmented Generation
Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a language model """
RAW_TEXT += """to produce grounded, accurate answers. Instead of relying solely on training data, RAG retrieves """
RAW_TEXT += """relevant documents from a knowledge base and includes them in the prompt. """
RAW_TEXT += """This allows the model to answer questions about unseen data, reduces hallucinations, """
RAW_TEXT += """and makes answers verifiable. The pipeline has two phases: indexing (embed and store) """
RAW_TEXT += """and querying (retrieve and generate).
The pipeline has two phases: indexing (embed and store) and querying (retrieve and generate)."""

# 2. The RAG prompt
rag_prompt = ChatPromptTemplate.from_template(
    'You are a helpful assistant.\n'
    'Answer the question using ONLY the context below.\n'
    'If the answer is not in the context say: '
    '"I do not have that information in my knowledge base."\n'
    'Do not make up information.\n\n'
    'Context:\n{context}\n\n'
    'Question: {question}'
)

# 3. The chain structured for direct text inputs
rag_chain = (
    {
        'context': lambda x: x['context'],
        'question': lambda x: x['question']
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 4. Invoke the chain using your RAW_TEXT
query = "What are the two phases of the pipeline?"
response = rag_chain.invoke({
    "context": RAW_TEXT,
    "question": query
})

print(f"Query: {query}\n")
print(f"Response:\n{response}")

Query: What are the two phases of the pipeline?

Response:
The two phases of the pipeline are: 

1. Indexing (embed and store)
2. Querying (retrieve and generate)
